In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Network Construction with LLM Annotation
=========================================
This script takes a JSON file containing KEGG pathway elements (generated by 
01_data_collection_kegg_api.py) and uses OpenAI's GPT model to annotate each 
pathway with cellular locations, functions, and interaction types, producing 
an XML representation. Finally, it builds a directed graph and saves the 
combined XML.

Usage:
    export OPENAI_API_KEY='your-key-here'
    python 02_network_construction_llm_annotation.py
"""

import os
import re
import time
import json
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor

import openai
import networkx as nx

# ----------------------------------------------------------------------
# 1. Define mapping dictionaries and prompt template
# ----------------------------------------------------------------------
interactions = {
    "activation": "->",
    "inhibition": "-|",
    "indirect_effect": "-->",
    "reciprocal_interaction": "<->",
    "conversion": "=>",
    "association": "//"
}

cellular_location_ref = {
    'EC': 'extracellular',
    'MB': 'membrane',
    'CY': 'cytoplasmic',
    'NU': 'nuclear',
    'MI': 'mitochondria',
    'MIM': 'mitochondria membrane',
    'MIP': 'mitochondria plasma'
}

body_location_ref = {
    'TOS': 'tissue/organ-specific: need specific annotations',
    'NTNOS': 'non-tissue & non-organ-specific',
    'UN': 'Unknown'
}

ppi_interactions = {
    'UREG': '''the process where a protein or signal enhances the expression,
               activity, or function of another protein, amplifying its biological effect.''',
    'DREGC': '''when a molecule or protein inhibits the activity or function of another
                by competing for the same binding site or resource,
                thereby reducing its biological effect''',
    'DREGNC': '''when a molecule or protein inhibits the activity or function of another
                 through mechanisms independent of direct competition,
                 such as allosteric modulation or signal interference''',
    'INTSB': '''where a molecule or protein binds to itself, often forming a dimer or oligomer.
                This interaction can influence the molecule's stability, function, or regulation,
                and is commonly seen in processes like protein folding, signal transduction,
                and enzyme regulation.''',
    'INTNSB': '''an interaction where a molecule or protein binds to a different molecule or protein,
                 rather than binding to itself. This type of interaction is crucial in many
                 biological processes, such as enzyme-substrate binding, receptor-ligand interactions,
                 and protein-protein interactions that drive cellular signaling, metabolic pathways,
                 and immune responses.''',
    'INTDIS': '''the process where a complex of molecules breaks apart into its individual components.''',
    'LIG': '''a molecule that binds to a receptor or protein, triggering a biological response'''
}

ntf_func_ref = {
    'ACT': 'activation',
    'INH': 'inhibition',
    'REG': 'regulator',
    'SIG': 'signal',
    'REC': '(signal) receptor',
    'TRD': 'Transducer',
    'EXE': 'executor'
}

tfc_func_ref = {
    'POS': 'Positive regulators',
    'REP': 'Repressor',
    'MOD': 'Modulator',
    'SCF': 'Scaffold'
}

sample_xml = """\
<pathway name="TCR signaling">
  <nodeList>
    <node id="node_1" name="CDKN2C">
      <cellularlocationList>
        <location name="NU" />
      </cellularlocationList>
      <bodylocationList>
        <location name="TOS" />
      </bodylocationList>
      <functionList>
        <function name="NTF-INH" />
        <function name="TFC-REP" />
      </functionList>
    </node>
    <node id="node_2" name="CDKN2A">
      <cellularlocationList>
        <location name="NU" />
      </cellularlocationList>
      <bodylocationList>
        <location name="TOS" />
      </bodylocationList>
      <functionList>
        <function name="NTF-INH" />
        <function name="TFC-REP" />
      </functionList>
    </node>
  </nodeList>
  <edgeList>
    <edge id="edge_1" type="UREG">
      <startNode node="TP53" />
      <endNode node="CDKN2A" />
    </edge>
    <edge id="edge_2" type="UREG">
      <startNode node="TP53" />
      <endNode node="CDKN1B" />
    </edge>
  </edgeList>
</pathway>
"""

user_prompt = f"""
Interaction information include:
Cellular location of protein: {cellular_location_ref},
Body Location of a protein: {body_location_ref}
interaction type: {ppi_interactions}
non-transcription factor functions are: {ntf_func_ref}
transcription factor and complexes functions are: {tfc_func_ref}

A sample output is like:
{sample_xml}
Output the formatted xml content for me.
"""

# ----------------------------------------------------------------------
# 2. Helper functions for API calls and XML processing
# ----------------------------------------------------------------------
def get_chatgpt_response(prompt, model="gpt-4o-mini-2024-07-18",
                         max_tokens=10000, temperature=0):
    """Call OpenAI Chat API with retry logic."""
    answered = False
    attempt = 1
    while not answered:
        try:
            response = openai.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful biological data processing assistant."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            answered = True
            print('one done')
            return response.choices[0].message.content.strip()
        except Exception as e:
            time.sleep(3)
            print(f"Request Error: {e}")
            print(f'Failed on attempt {attempt}')
            attempt += 1
            if attempt > 3:
                answered = True
                return ''


def get_xml_data(pathway):
    """Send pathway definition to GPT and return XML."""
    time.sleep(1)  # polite rate limiting
    prompt = f"""
I want you to process some data for me. I will give you a biological pathway,
and you will put it into a xml-format output.
I will tell you the pathway names and involved proteins:
{pathway['name']}, {pathway['definition']}.
"""
    full_prompt = prompt + user_prompt
    reply = get_chatgpt_response(full_prompt)
    return reply


def clean_xml(raw_xml):
    """Remove markdown code fences and extract pure XML."""
    try:
        # Find first '<' after possible ```xml or ```
        start = '<' + raw_xml.split('<', 1)[1].strip("```").strip('xml')
        # Find last '>'
        ind = start.rfind('>')
        return start[:ind + 1]
    except Exception:
        return raw_xml


def build_network(xml_data, graph):
    """Parse XML and add nodes/edges to a NetworkX DiGraph."""
    root = ET.fromstring(xml_data)
    # Add nodes
    for node in root.find('nodeList'):
        node_id = node.attrib['id']
        node_name = node.attrib['name']
        cellular_location = node.find('cellularlocationList/location').attrib['name']
        function = node.find('functionList/function').attrib['name']
        graph.add_node(node_name,
                       cellular_location=cellular_location,
                       function=function)
    # Add edges
    for edge in root.find('edgeList'):
        start_node = edge.find('startNode').attrib['node']
        end_node = edge.find('endNode').attrib['node']
        edge_type = edge.attrib['type']
        graph.add_edge(start_node, end_node, edge_type=edge_type)


# ----------------------------------------------------------------------
# 3. Main processing pipeline
# ----------------------------------------------------------------------
def main():
    # Read OpenAI API key from environment variable
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Please set the OPENAI_API_KEY environment variable.")
    openai.api_key = api_key

    # Load input JSON (generated by 01_data_collection_kegg_api.py)
    input_file = "oncogene_ppi.json"
    if not os.path.exists(input_file):
        print(f"Input file {input_file} not found. Please run the data collection step first.")
        return

    with open(input_file, 'r') as f:
        pathway_data = json.load(f)

    ppi_dict_list = list(pathway_data.values())
    print(f"Loaded {len(ppi_dict_list)} pathway entries.")

    # Process in parallel (adjust max_workers as needed)
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(executor.map(get_xml_data, ppi_dict_list))

    # Filter out None results and clean XML
    filtered_list = [clean_xml(item) for item in results if item is not None]

    # Build a directed graph
    G = nx.DiGraph()
    success_count = 0
    for xml_str in filtered_list:
        try:
            build_network(xml_str, G)
            success_count += 1
        except Exception as e:
            print(f"Failed to parse XML: {e}")
            continue
    print(f"{success_count} pathways successfully integrated into graph.")

    # Remove nodes that are ions (often appear as artifacts)
    nodes_to_remove = [node for node in G.nodes() if "Ca2+" in node or "K+" in node or "Na+" in node]
    G.remove_nodes_from(nodes_to_remove)

    # Expand composite nodes (e.g., "A+B") into individual components
    composite_nodes = [node for node in G.nodes() if '+' in node]
    for comp in composite_nodes:
        parts = comp.split('+')
        for part in parts:
            if not G.has_node(part):
                G.add_node(part)
            G.add_edge(part, comp, edge_type='combine')
    G.remove_nodes_from(nodes_to_remove)  # ensure ions removed again

    # Combine all XML fragments into one document
    root = ET.Element("pathways")
    for xml_content in filtered_list:
        try:
            tree = ET.ElementTree(ET.fromstring(xml_content))
            root.append(tree.getroot())
        except Exception as e:
            print(f"Skipping invalid XML: {e}")
    combined_tree = ET.ElementTree(root)
    output_file = "oncogene_pathways.xml"
    combined_tree.write(output_file, encoding='utf-8', xml_declaration=True)
    print(f"Saved combined XML to {output_file}")


if __name__ == "__main__":
    main()

ValueError: Please set the OPENAI_API_KEY environment variable.